In [1]:
# ======================================================================
# 06_metrics.ipynb
# FinGap-Governance | Aggregate run records into the reviewer-requested
# empirical metrics (these are now RESULTS, not proposed metrics):
#   * determination accuracy                  (per task x condition)
#   * traceability completion rate            (per task x condition)
#   * correct-but-untraceable rate            (the paper's core quantity)
#   * failure-type profile                    (per condition, over tasks)
#   * boundary-misjudgment detection rate     (boundary cases only)
#   * cross-task failure-profile similarity   (the "isomorphism" metric)
# Variability is reported as mean +/- std across the N_REPEATS runs.
# Outputs -> results/tables/metrics_*.csv
# ======================================================================
from fingap_core import *
import pandas as pd, numpy as np, json

df = pd.concat([pd.read_json(TABLES / f"{t}_runs.json") for t in TASKS
                if (TABLES / f"{t}_runs.json").exists()], ignore_index=True)
assert len(df), "no run records found -- run notebooks 03-05 first"

# ---- per-repeat aggregation so we can report mean +/- std -------------
def per_repeat(metric):
    g = (df.groupby(["task", "condition", "run_index"])[metric].mean()
           .groupby(level=[0, 1]).agg(["mean", "std"]).reset_index())
    g["metric"] = metric
    return g

primary = pd.concat([per_repeat(m) for m in
                     ["determination_correct", "traceable", "correct_but_untraceable"]])
primary[["mean", "std"]] = (primary[["mean", "std"]] * 100).round(1)
primary = primary.pivot_table(index=["task", "condition"], columns="metric",
                              values=["mean", "std"]).round(1)
primary.to_csv(TABLES / "metrics_primary.csv")
print("=== primary metrics (%) — mean over repeats ===")
print(primary["mean"].to_string())

# ---- boundary-misjudgment detection rate ------------------------------
bd = (df[df.difficulty == "boundary"].groupby(["task", "condition"])["determination_correct"]
        .mean().mul(100).round(1).reset_index(name="boundary_detection_rate_pct"))
bd.to_csv(TABLES / "metrics_boundary.csv", index=False)
print("\n=== boundary-misjudgment detection rate (%) ===")
print(bd.to_string(index=False))

# ---- failure-type profile per condition -------------------------------
prof = (df.dropna(subset=["failure"]).groupby(["condition", "failure"]).size()
          .unstack(fill_value=0).reindex(columns=FAILURE_TYPES, fill_value=0))
prof_prop = prof.div(prof.sum(axis=1), axis=0).round(3)
prof.to_csv(TABLES / "metrics_failure_counts.csv")
prof_prop.to_csv(TABLES / "metrics_failure_profile.csv")
print("\n=== failure-type profile by condition (proportions) ===")
print(prof_prop.to_string())

# ---- cross-task failure-profile similarity (the isomorphism metric) ---
# For each condition, build task x failure-type proportion vectors and take
# pairwise cosine similarity across tasks; high values = the SAME failure
# structure recurs across tasks (structural recurrence / "isomorphism").
def cos(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return float(a @ b / ((np.linalg.norm(a) * np.linalg.norm(b)) or 1))

iso_rows = []
for cond in CONDITIONS:
    sub = df[(df.condition == cond)].dropna(subset=["failure"])
    vecs = {}
    for t in TASKS:
        v = (sub[sub.task == t]["failure"].value_counts()
             .reindex(FAILURE_TYPES, fill_value=0)).values
        vecs[t] = v
    pairs = [("advisory", "anomaly"), ("advisory", "accounting"), ("anomaly", "accounting")]
    for a, b in pairs:
        iso_rows.append({"condition": cond, "task_pair": f"{a}~{b}",
                         "cosine_similarity": round(cos(vecs[a], vecs[b]), 3)})
iso = pd.DataFrame(iso_rows)
iso.to_csv(TABLES / "metrics_isomorphism.csv", index=False)
print("\n=== cross-task failure-profile similarity (cosine) ===")
print(iso.pivot(index="task_pair", columns="condition", values="cosine_similarity").to_string())

print("\nsaved -> results/tables/metrics_*.csv")


=== primary metrics (%) — mean over repeats ===
metric                  correct_but_untraceable  determination_correct  traceable
task       condition                                                             
accounting direct                          76.7                   76.7        0.0
           graph_rag                       20.4                   88.5       68.1
           vanilla_rag                     71.5                   83.7       12.2
advisory   direct                          52.6                   52.6        0.0
           graph_rag                       16.3                   95.9       79.6
           vanilla_rag                     68.5                   68.5        0.0
anomaly    direct                          73.0                   73.0        0.0
           graph_rag                        1.9                   98.1       96.3
           vanilla_rag                     77.0                   78.1        1.1

=== boundary-misjudgment detection rate (%) ===
 